In [2]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report

In [5]:
from pathlib import Path

# Load and clean the data (norm)

In [3]:
def norm(s):
    s = str(s).lower()
    s = re.sub(r'\(.*?\)', '', s)          # strip equipment notes in parens
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

In [6]:
# 1. Define the base directory generically
DATA_DIR = Path('C:/Users/iamph/OneDrive/Documents/Datasets/Kaggle_Fitness_Workout_Dataset')

In [8]:
labeled = pd.read_csv(DATA_DIR / 'exercises_dataset_CLEAN.csv')
labeled['norm'] = labeled['exercise'].apply(norm)

prog = pd.read_csv(DATA_DIR / 'programs_detailed_boostcamp_kaggle_CLEAN2.csv')
unique_names = pd.DataFrame({'exercise_name': prog['exercise_name'].unique()})
unique_names['norm'] = unique_names['exercise_name'].apply(norm)

In [11]:
prog.shape

(261897, 16)

In [12]:
prog['exercise_name'].unique()

<StringArray>
[                 'Squat (Barbell)',           'Leg Press (45 Degrees)',
                    'Leg Extension',          'Preacher Curl (Barbell)',
          'Bent Over Row (Barbell)',               'Single Arm Iso Row',
             'Pull-Up (Bodyweight)', 'V-Handle Tricep Pushdown (Cable)',
            'Bench Press (Barbell)',    'Incline Bench Press (Barbell)',
 ...
    'Bottom-Half Seated Cable Flye',                       'T Bar Rows',
    'Seated Barbell Shoulder Press',         'Lat Pulldown Netural TMJ',
            'Kneeling Lat Pulldown',         'Single Arm Rear Delt Fly',
                'Sandbag Box Squat',            'Sandbag Bent-Over Row',
              'Sandbag Floor Press',               'Inverted Face Pull']
Length: 1826, dtype: str

In [9]:
labeled

,exercise,label,norm
0,3/4 sit-up,Abs/Core,3 4 sit up
1,45° side bend,Abs/Core,45 side bend
2,air bike,Abs/Core,air bike
3,all fours squad stretch,Legs,all fours squad stretch
4,alternate heel touchers,Abs/Core,alternate heel touchers
...,...,...,...
4237,EZ-bar skullcrusher-,Arms,ez bar skullcrusher
4238,Lying Close-Grip Barbell Triceps Press To Chin,Arms,lying close grip barbell triceps press to chin
4239,EZ-Bar Skullcrusher - Gethin Variation,Arms,ez bar skullcrusher gethin variation
4240,TBS Skullcrusher,Arms,tbs skullcrusher


In [10]:
unique_names

,exercise_name,norm
0,Squat (Barbell),squat
1,Leg Press (45 Degrees),leg press
2,Leg Extension,leg extension
3,Preacher Curl (Barbell),preacher curl
4,Bent Over Row (Barbell),bent over row
...,...,...
1821,Single Arm Rear Delt Fly,single arm rear delt fly
1822,Sandbag Box Squat,sandbag box squat
1823,Sandbag Bent-Over Row,sandbag bent over row
1824,Sandbag Floor Press,sandbag floor press


# Layer 1: exact-match bootstrap

In [13]:
# Layer 1: exact-match bootstrap
lookup = labeled.drop_duplicates('norm').set_index('norm')['label']

In [14]:
lookup

norm
3 4 sit up                                        Abs/Core
45 side bend                                      Abs/Core
air bike                                          Abs/Core
all fours squad stretch                               Legs
alternate heel touchers                           Abs/Core
                                                    ...   
ez bar skullcrusher                                   Arms
lying close grip barbell triceps press to chin        Arms
ez bar skullcrusher gethin variation                  Arms
tbs skullcrusher                                      Arms
30 arms ez bar skullcrusher                           Arms
Name: label, Length: 4001, dtype: str

In [15]:
unique_names['label'] = unique_names['norm'].map(lookup)
to_predict = unique_names[unique_names['label'].isna()].copy()
print(f"{unique_names['label'].notna().sum()} matched directly, {len(to_predict)} need prediction")

295 matched directly, 1531 need prediction


In [16]:
unique_names

,exercise_name,norm,label
0,Squat (Barbell),squat,NaN
1,Leg Press (45 Degrees),leg press,Legs
2,Leg Extension,leg extension,Legs
3,Preacher Curl (Barbell),preacher curl,Arms
4,Bent Over Row (Barbell),bent over row,NaN
...,...,...,...
1821,Single Arm Rear Delt Fly,single arm rear delt fly,NaN
1822,Sandbag Box Squat,sandbag box squat,NaN
1823,Sandbag Bent-Over Row,sandbag bent over row,NaN
1824,Sandbag Floor Press,sandbag floor press,NaN


# Layer 2: Train classifier on labeled set

In [17]:
vectorizer = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1,2), min_df=1, stop_words="english")),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=1)),
])
pipe = Pipeline([("feats", vectorizer), ("clf", LinearSVC(class_weight="balanced", max_iter=5000))])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, labeled['exercise'], labeled['label'], cv=cv, scoring='f1_macro', n_jobs=-1)
print("CV macro-F1:", scores.mean(), scores.std())

pipe.fit(labeled['exercise'], labeled['label'])
to_predict['label'] = pipe.predict(to_predict['exercise_name'])

CV macro-F1: 0.7400362459617755 0.02123212270243666


In [18]:
scores

array([0.7663953 , 0.70665323, 0.7255511 , 0.7521416 , 0.74944   ])

## Confidence for manual-review triage

In [19]:
# confidence for manual-review triage
margins = pipe.decision_function(to_predict['exercise_name'])
to_predict['confidence'] = margins.max(axis=1) - sorted(margins, key=lambda r: -r[0])[0][0] if False else None


In [24]:
# simpler: margin between top-2 classes
import numpy as np
sorted_margins = np.sort(margins, axis=1)
to_predict['confidence'] = sorted_margins[:, -1] - sorted_margins[:, -2]
  # review these first

,exercise_name,norm,label,confidence
716,Competition (Placeholder),competition,Abs/Core,0.001476
469,Deadlift (Deficit),deadlift,Legs,0.002527
626,Single Leg / Core Accessories,single leg core accessories,Abs/Core,0.002770
985,Sled Drive,sled drive,Legs,0.003811
1210,Loop band Exercise,loop band exercise,Back,0.004218
1816,Bottom-Half Seated Cable Flye,bottom half seated cable flye,Shoulders/Neck,0.004220
343,Pin Press Bench (Barbell),pin press bench,Chest,0.005322
1460,Backward Sled,backward sled,Shoulders/Neck,0.005470
1372,Resisted Rotations,resisted rotations,Abs/Core,0.005554
1040,Loading,loading,Back,0.006435


In [31]:
to_predict.sort_values('confidence').head(30)

,exercise_name,norm,label,confidence
716,Competition (Placeholder),competition,Abs/Core,0.001476
469,Deadlift (Deficit),deadlift,Legs,0.002527
626,Single Leg / Core Accessories,single leg core accessories,Abs/Core,0.002770
985,Sled Drive,sled drive,Legs,0.003811
1210,Loop band Exercise,loop band exercise,Back,0.004218
1816,Bottom-Half Seated Cable Flye,bottom half seated cable flye,Shoulders/Neck,0.004220
343,Pin Press Bench (Barbell),pin press bench,Chest,0.005322
1460,Backward Sled,backward sled,Shoulders/Neck,0.005470
1372,Resisted Rotations,resisted rotations,Abs/Core,0.005554
1040,Loading,loading,Back,0.006435


In [28]:
to_predict.sort_values('confidence').tail(30)

,exercise_name,norm,label,confidence
1031,Hammer Squat Lung Calf Raises,hammer squat lung calf raises,Legs,3.379435
0,Squat (Barbell),squat,Legs,3.411159
1332,Customisable Row,customisable row,Back,3.464481
1458,Sumo Squat Belt Squat,sumo squat belt squat,Legs,3.472986
1743,Split Squat,split squat,Legs,3.483531
614,Platz Squat,platz squat,Legs,3.551641
789,Batwing Row,batwing row,Back,3.627135
444,Sumo Box Squat,sumo box squat,Legs,3.635805
627,Kroc Row,kroc row,Back,3.640249
467,Leg Press Calf,leg press calf,Legs,3.662958


## Calibrated probabilities to estimate errors

In [32]:
from sklearn.calibration import CalibratedClassifierCV

calibrated = CalibratedClassifierCV(pipe, cv=5, method='sigmoid')
calibrated.fit(labeled['exercise'], labeled['label'])

probs = calibrated.predict_proba(to_predict['exercise_name'])
to_predict['confidence'] = probs.max(axis=1)
to_predict.sort_values('confidence').head(30)  # review these first

,exercise_name,norm,label,confidence
679,Cable Forearm Shit,cable forearm shit,Stretching,0.200470
408,Weight Over Bar (Highland Games),weight over bar,Arms,0.201660
523,PullAround,pullaround,Back,0.214238
713,Weight Over Bar,weight over bar,Back,0.215565
880,Dynamic Warm-up,dynamic warm up,Shoulders/Neck,0.223600
776,Kneeling Landmine Pushes,kneeling landmine pushes,Back,0.236658
1504,Cycling Cardio,cycling cardio,Cardio/Plyos,0.237466
1280,Dumbbell Swings,dumbbell swings,Arms,0.239759
1188,Ladder Drills,ladder drills,Cardio/Plyos,0.240629
564,Cycling,cycling,Cardio/Plyos,0.243867


In [36]:
to_predict.shape

(1531, 4)

In [40]:
to_predict[to_predict.confidence < 0.7].shape

(722, 4)

In [34]:
to_predict.sort_values('confidence').tail(20)

,exercise_name,norm,label,confidence
1458,Sumo Squat Belt Squat,sumo squat belt squat,Legs,0.963416
88,Squat (Dumbbell),squat,Legs,0.966032
1361,MTS Row,mts row,Back,0.966436
672,100m Row,100m row,Back,0.966914
1690,10km Row,10km row,Back,0.966914
512,Dumbbell Spanish Squat,dumbbell spanish squat,Legs,0.968554
614,Platz Squat,platz squat,Legs,0.969968
1474,Precor Squat,precor squat,Legs,0.971497
1734,300m Row,300m row,Back,0.972654
622,1km Row,1km row,Back,0.974448


In [38]:
to_predict.sort_values('confidence').to_csv('values_to_predict.csv')

# Load manually labelled data

In [43]:
predicted = pd.read_csv(DATA_DIR  / 'exercises_labelled.csv')

In [44]:
predicted

,Column1,exercise_name,norm,label,confidence,adjusted_label,claude_label,new_label
0,679,Cable Forearm Shit,cable forearm shit,Stretching,20.00%,Arms,Arms,Arms
1,408,Weight Over Bar (Highland Games),weight over bar,Arms,20.20%,Legs,Cardio/Plyometrics,Legs
2,523,PullAround,pullaround,Back,21.40%,NaN,Back,Back
3,713,Weight Over Bar,weight over bar,Back,21.60%,Legs,Cardio/Plyometrics,Legs
4,880,Dynamic Warm-up,dynamic warm up,Shoulders/Neck,22.40%,Cardio/Plyos,Stretching,Cardio/Plyos
...,...,...,...,...,...,...,...,...
1526,1813,Row,row,Back,97.40%,NaN,NaN,Back
1527,1736,400m Row,400m row,Back,97.40%,NaN,NaN,Back
1528,409,Belt Squat,belt squat,Legs,97.50%,NaN,NaN,Legs
1529,134,SSB Squat,ssb squat,Legs,98.00%,NaN,NaN,Legs


In [45]:
unique_names

,exercise_name,norm,label
0,Squat (Barbell),squat,NaN
1,Leg Press (45 Degrees),leg press,Legs
2,Leg Extension,leg extension,Legs
3,Preacher Curl (Barbell),preacher curl,Arms
4,Bent Over Row (Barbell),bent over row,NaN
...,...,...,...
1821,Single Arm Rear Delt Fly,single arm rear delt fly,NaN
1822,Sandbag Box Squat,sandbag box squat,NaN
1823,Sandbag Bent-Over Row,sandbag bent over row,NaN
1824,Sandbag Floor Press,sandbag floor press,NaN


In [46]:
unique_names.isnull().sum()

exercise_name       0
norm                0
label            1531
dtype: int64

In [50]:
unique_names

,exercise_name,norm,label
0,Squat (Barbell),squat,NaN
1,Leg Press (45 Degrees),leg press,Legs
2,Leg Extension,leg extension,Legs
3,Preacher Curl (Barbell),preacher curl,Arms
4,Bent Over Row (Barbell),bent over row,NaN
...,...,...,...
1821,Single Arm Rear Delt Fly,single arm rear delt fly,NaN
1822,Sandbag Box Squat,sandbag box squat,NaN
1823,Sandbag Bent-Over Row,sandbag bent over row,NaN
1824,Sandbag Floor Press,sandbag floor press,NaN


In [51]:
for row in predicted.itertuples():
    idx = unique_names[unique_names.norm == row.norm].index
    unique_names.loc[idx,'label'] = row.new_label

In [52]:
unique_names

,exercise_name,norm,label
0,Squat (Barbell),squat,Legs
1,Leg Press (45 Degrees),leg press,Legs
2,Leg Extension,leg extension,Legs
3,Preacher Curl (Barbell),preacher curl,Arms
4,Bent Over Row (Barbell),bent over row,Back
...,...,...,...
1821,Single Arm Rear Delt Fly,single arm rear delt fly,Shoulders/Neck
1822,Sandbag Box Squat,sandbag box squat,Legs
1823,Sandbag Bent-Over Row,sandbag bent over row,Back
1824,Sandbag Floor Press,sandbag floor press,Chest


## Create a dictionary for exercise -> label

In [53]:
exercise_label = dict(zip(unique_names['exercise_name'], unique_names['label']))

In [54]:
exercise_label

{'Squat (Barbell)': 'Legs',
 'Leg Press (45 Degrees)': 'Legs',
 'Leg Extension': 'Legs',
 'Preacher Curl (Barbell)': 'Arms',
 'Bent Over Row (Barbell)': 'Back',
 'Single Arm Iso Row': 'Back',
 'Pull-Up (Bodyweight)': 'Back',
 'V-Handle Tricep Pushdown (Cable)': 'Arms',
 'Bench Press (Barbell)': 'Chest',
 'Incline Bench Press (Barbell)': 'Chest',
 'Pec Fly (Dumbbell)': 'Chest',
 'Deadlift (Barbell)': 'Legs',
 'Front Squat (Barbell)': 'Legs',
 'Bulgarian Split Squat (Dumbbell)': 'Legs',
 'Overhead Press (Barbell)': 'Shoulders/Neck',
 'Lateral Raise (Dumbbell)': 'Shoulders/Neck',
 'Incline Bench Press (Dumbbell)': 'Chest',
 'Barbell Row': 'Back',
 'Lat Pulldown': 'Back',
 'Face Pull': 'Shoulders/Neck',
 'Squat (Low Bar)': 'Legs',
 'Romanian Deadlift (Barbell)': 'Legs',
 'Leg Press': 'Legs',
 'Seated Hamstring Curl': 'Legs',
 'Calf Raise (Machine)': 'Legs',
 'Bicep Curl (Barbell)': 'Arms',
 'Hammer Curl (Cable)': 'Arms',
 'Bench Press (Close Grip)': 'Chest',
 'Overhead Tricep Extension (Ca

# Label the original dataset

In [55]:
prog

,title,description,level,goal,equipment,program_length,time_per_workout,week,day,number_of_exercises,exercise_name,sets,reps,intensity,created,last_edit
0,nSuns_Rebuild by KSI,The purpose of this program is to build streng...,"{'Beginner', 'Advanced', 'Intermediate'}","{'Bodybuilding', 'Powerbuilding'}",Full Gym,7.0,90.0,1.0,1.0,4.0,Squat (Barbell),1.0,15.0,8.0,2025-04-06 03:06:00,2025-06-18 08:29:00
1,nSuns_Rebuild by KSI,The purpose of this program is to build streng...,"{'Beginner', 'Advanced', 'Intermediate'}","{'Bodybuilding', 'Powerbuilding'}",Full Gym,7.0,90.0,1.0,1.0,4.0,Leg Press (45 Degrees),1.0,100.0,9.0,2025-04-06 03:06:00,2025-06-18 08:29:00
2,nSuns_Rebuild by KSI,The purpose of this program is to build streng...,"{'Beginner', 'Advanced', 'Intermediate'}","{'Bodybuilding', 'Powerbuilding'}",Full Gym,7.0,90.0,1.0,1.0,4.0,Leg Extension,4.0,5.0,9.0,2025-04-06 03:06:00,2025-06-18 08:29:00
3,nSuns_Rebuild by KSI,The purpose of this program is to build streng...,"{'Beginner', 'Advanced', 'Intermediate'}","{'Bodybuilding', 'Powerbuilding'}",Full Gym,7.0,90.0,1.0,1.0,4.0,Preacher Curl (Barbell),1.0,10.0,9.0,2025-04-06 03:06:00,2025-06-18 08:29:00
4,nSuns_Rebuild by KSI,The purpose of this program is to build streng...,"{'Beginner', 'Advanced', 'Intermediate'}","{'Bodybuilding', 'Powerbuilding'}",Full Gym,7.0,90.0,1.0,2.0,4.0,Bent Over Row (Barbell),1.0,15.0,9.0,2025-04-06 03:06:00,2025-06-18 08:29:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261892,3 Day Baki Grappler Condensed Conjugate (2),This program is designed for martial artists w...,"{'Advanced', 'Intermediate'}","{'Bodybuilding', 'Muscle & Sculpting', 'Athlet...",Garage Gym,4.0,60.0,4.0,3.0,5.0,Overhead Press (Barbell),3.0,1.0,9.0,2025-06-21 03:42:00,2025-06-30 01:20:00
261893,3 Day Baki Grappler Condensed Conjugate (2),This program is designed for martial artists w...,"{'Advanced', 'Intermediate'}","{'Bodybuilding', 'Muscle & Sculpting', 'Athlet...",Garage Gym,4.0,60.0,4.0,3.0,5.0,Barbell Row,3.0,1.0,9.0,2025-06-21 03:42:00,2025-06-30 01:20:00
261894,3 Day Baki Grappler Condensed Conjugate (2),This program is designed for martial artists w...,"{'Advanced', 'Intermediate'}","{'Bodybuilding', 'Muscle & Sculpting', 'Athlet...",Garage Gym,4.0,60.0,4.0,3.0,5.0,Zercher Squat (Barbell),6.0,10.0,9.0,2025-06-21 03:42:00,2025-06-30 01:20:00
261895,3 Day Baki Grappler Condensed Conjugate (2),This program is designed for martial artists w...,"{'Advanced', 'Intermediate'}","{'Bodybuilding', 'Muscle & Sculpting', 'Athlet...",Garage Gym,4.0,60.0,4.0,3.0,5.0,Upright Row (Barbell),4.0,1.0,9.0,2025-06-21 03:42:00,2025-06-30 01:20:00


In [56]:
prog['exercise_label'] = prog['exercise_name'].map(exercise_label)

In [57]:
prog.to_csv('programs_detailed_boostcamp_kaggle_LABELLED.csv')